In [2]:
import pandas as pd
from neo4j import GraphDatabase

URI = "neo4j+s://d03ca05f.databases.neo4j.io"
USERNAME = "d03ca05f"
PASSWORD = "EYeif4KyCsqUpd4FCEnJYANV1-dqGfv6j0suV8ckg-8"

CSV_FILE = "../../data/sepolia_transactions.csv"


driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)


def create_graph(tx, rows):
    query = """
    UNWIND $rows AS row

    MERGE (sender:Wallet {
        address: row.from
    })

    MERGE (receiver:Wallet {
        address: row.to
    })

    CREATE (sender)-[t:SENT {
        tx_hash: row.tx_hash,
        value: row.value,
        block_number: row.block_number
    }]->(receiver)
    """

    tx.run(query, rows=rows)


df = pd.read_csv(CSV_FILE)

# Convert dataframe into dictionaries
rows = df.to_dict("records")

with driver.session() as session:

    # Import in batches
    batch_size = 500

    for i in range(0, len(rows), batch_size):

        batch = rows[i:i + batch_size]

        session.execute_write(
            create_graph,
            batch
        )

        print(
            f"Imported {min(i + batch_size, len(rows))}/{len(rows)} transactions"
        )


driver.close()

print("Import complete!")

ClientError: {neo4j_code: Neo.ClientError.Statement.SemanticError} {message: Cannot merge the following node because of NaN property value for 'address': (:Wallet {address: NaN})} {gql_status: 22G03} {gql_status_description: error: data exception - invalid value type}

In [5]:
import pandas as pd

df = pd.read_csv("../../data/sepolia_transactions.csv")

print("Total rows:", len(df))

print("\nMissing values:")
print(df.isna().sum())

Total rows: 2618

Missing values:
tx_hash         0
from            0
to              7
value           0
block_number    0
dtype: int64


In [6]:
df[df["to"].isna()]

,tx_hash,from,to,value,block_number
6,0xd742ea9e20cd07c88dd9e06a0367710d01c2e24fe9c9...,0xea3a54b28fdd497c2861ba0ef3827d4bc324ceae,NaN,0x0,11645660
100,0x8b035dd4114ed292ccd9ad523c75a5175ab2f5dc8fb5...,0xea3a54b28fdd497c2861ba0ef3827d4bc324ceae,NaN,0x0,11645661
764,0x67692785534fbded491a515a9bea88ee797bc9e8b340...,0xb82194c3a8733bd16e5c62786ccbc592a874494c,NaN,0x0,11645667
1102,0xe06b100bbbc886ddbcfa20dec830d6813ba1b19b92ab...,0x4910482523300a90b07195dd512151770b935418,NaN,0x0,11645670
1626,0x08d8d4389755f64ba959fabeb15dff10ae3945efb68b...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674
1627,0xa3ee2b074ce09da59adbd2cc1622a4ca046a0c72a039...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674
1638,0x2757afc0d863936e906203133622741d40011ec38f43...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674


In [7]:
df.loc[df["to"].isna(), ["tx_hash", "from", "to", "value", "block_number"]]

,tx_hash,from,to,value,block_number
6,0xd742ea9e20cd07c88dd9e06a0367710d01c2e24fe9c9...,0xea3a54b28fdd497c2861ba0ef3827d4bc324ceae,NaN,0x0,11645660
100,0x8b035dd4114ed292ccd9ad523c75a5175ab2f5dc8fb5...,0xea3a54b28fdd497c2861ba0ef3827d4bc324ceae,NaN,0x0,11645661
764,0x67692785534fbded491a515a9bea88ee797bc9e8b340...,0xb82194c3a8733bd16e5c62786ccbc592a874494c,NaN,0x0,11645667
1102,0xe06b100bbbc886ddbcfa20dec830d6813ba1b19b92ab...,0x4910482523300a90b07195dd512151770b935418,NaN,0x0,11645670
1626,0x08d8d4389755f64ba959fabeb15dff10ae3945efb68b...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674
1627,0xa3ee2b074ce09da59adbd2cc1622a4ca046a0c72a039...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674
1638,0x2757afc0d863936e906203133622741d40011ec38f43...,0xddb968e5d31fd578115096f1e2be33bdb7f348b2,NaN,0x0,11645674


In [8]:
df = df.dropna(subset=["to"]).copy()

print("Transactions to import:", len(df))

Transactions to import: 2611


In [9]:
rows = df.to_dict("records")

In [10]:
import pandas as pd
from neo4j import GraphDatabase

URI = "neo4j+s://d03ca05f.databases.neo4j.io"
USERNAME = "d03ca05f"
PASSWORD = "EYeif4KyCsqUpd4FCEnJYANV1-dqGfv6j0suV8ckg-8"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)


def create_graph(tx, rows):
    query = """
    UNWIND $rows AS row

    MERGE (sender:Wallet {
        address: row.from
    })

    MERGE (receiver:Wallet {
        address: row.to
    })

    CREATE (sender)-[t:SENT {
        tx_hash: row.tx_hash,
        value: row.value,
        block_number: row.block_number
    }]->(receiver)
    """

    tx.run(query, rows=rows)
    
rows = df.to_dict("records")

with driver.session() as session:

    # Import in batches
    batch_size = 500

    for i in range(0, len(rows), batch_size):

        batch = rows[i:i + batch_size]

        session.execute_write(
            create_graph,
            batch
        )

        print(
            f"Imported {min(i + batch_size, len(rows))}/{len(rows)} transactions"
        )


driver.close()

print("Import complete!")

Imported 500/2611 transactions
Imported 1000/2611 transactions
Imported 1500/2611 transactions
Imported 2000/2611 transactions
Imported 2500/2611 transactions
Imported 2611/2611 transactions
Import complete!
